# LeetCode #208 — Implement Trie (Prefix Tree): cây tiền tố

**Mức độ:** Medium · **Chủ đề:** Hash Table, String, Design, Trie

Skeleton trên LeetCode:

```python
class Trie:
    def __init__(self):
    def insert(self, word: str) -> None:
    def search(self, word: str) -> bool:
    def startsWith(self, prefix: str) -> bool:
```

Mục tiêu của notebook:

1. Hiểu **trie**: cây mà mỗi cạnh là **một ký tự**, mỗi đường đi từ gốc là một **tiền tố**.
2. Vì sao cần cờ **`is_end`**: phân biệt "có từ `app`" với "chỉ có tiền tố `app` của `apple`".
3. Cài đặt `insert`, `search`, `startsWith` — đều `O(độ dài từ)`, không phụ thuộc số từ đã lưu.
4. Hai cách lưu con: `dict` hoặc mảng 26 phần tử; mở rộng: đếm số từ có tiền tố, xóa từ, gợi ý tự động.

# 1. Bài toán là gì?

Cài đặt lớp `Trie`:

- `insert(word)`: thêm `word` vào trie.
- `search(word)`: `True` nếu `word` **đã được thêm** (nguyên từ).
- `startsWith(prefix)`: `True` nếu **có từ nào** đã thêm bắt đầu bằng `prefix`.

**Constraints (theo đề):**

- `1 <= word.length, prefix.length <= 2000`
- `word`, `prefix` chỉ gồm chữ cái thường.
- Tối đa `3 * 10^4` lời gọi.

---

## Ví dụ

```text
insert("apple")
search("apple")     -> True
search("app")       -> False     ("app" chỉ là tiền tố, chưa được thêm như một từ)
startsWith("app")   -> True
insert("app")
search("app")       -> True
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Trie (cây tiền tố)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Trie (cây tiền tố)

### Trie là gì?

Trie là cây trong đó mỗi **cạnh** mang **một ký tự**; đường đi từ gốc tới một nút tạo thành một **tiền tố**. Các từ có chung tiền tố **dùng chung** đường đi. Mỗi nút có cờ **kết thúc** cho biết tiền tố đó có phải một từ đã thêm không.

```text
thêm: "car", "cat", "cart", "dog"

          (gốc)
          /   \
         c     d
         |     |
         a     o
        / \    |
       r*  t*  g*
       |
       t*            * = kết thúc một từ
```

### Ba thao tác cơ bản — đều O(độ dài chuỗi)

In [1]:
class TrieNodeDemo:
    def __init__(self):
        self.children = {}          # ký tự → nút con
        self.end = False


def trie_insert(root, word):
    node = root
    for ch in word:
        if ch not in node.children:
            node.children[ch] = TrieNodeDemo()
        node = node.children[ch]
    node.end = True


def trie_walk(root, s):
    node = root
    for ch in s:
        node = node.children.get(ch)
        if node is None:
            return None
    return node


root_demo = TrieNodeDemo()
for w in ["car", "cat", "cart", "dog"]:
    trie_insert(root_demo, w)

for q in ["car", "ca", "cart", "cab"]:
    nd = trie_walk(root_demo, q)
    print(f"{q!r:7}: là tiền tố? {str(nd is not None):<5} | là từ? {nd is not None and nd.end}")

'car'  : là tiền tố? True  | là từ? True
'ca'   : là tiền tố? True  | là từ? False
'cart' : là tiền tố? True  | là từ? True
'cab'  : là tiền tố? False | là từ? False


### Liệt kê mọi từ có một tiền tố (gợi ý tự động)

Đi tới nút của tiền tố, rồi **DFS** toàn bộ cây con.

In [2]:
def words_with_prefix(root, prefix):
    start = trie_walk(root, prefix)
    out = []
    def dfs(node, path):
        if node.end:
            out.append(path)
        for ch in sorted(node.children):
            dfs(node.children[ch], path + ch)
    if start:
        dfs(start, prefix)
    return out

print(words_with_prefix(root_demo, "ca"), words_with_prefix(root_demo, "x"))

['car', 'cart', 'cat'] []


### Cài đặt gọn bằng dict lồng nhau

Mỗi nút là một `dict`; dùng một khóa đặc biệt (ví dụ `"$"`, không phải chữ cái) làm cờ kết thúc.

In [3]:
root_d = {}
for w in ["car", "cat"]:
    node = root_d
    for ch in w:
        node = node.setdefault(ch, {})
    node["$"] = True
print(root_d)

{'c': {'a': {'r': {'$': True}, 't': {'$': True}}}}


### So sánh với `set` các từ

| Thao tác | `set` | Trie |
|---|---|---|
| có từ `w`? | O(L) trung bình | O(L) |
| có từ nào bắt đầu bằng `p`? | O(N · L) — duyệt mọi từ | **O(len(p))** |
| liệt kê từ theo tiền tố, thứ tự từ điển | sắp xếp lại | DFS theo thứ tự con |

### Lỗi hay gặp

- Không có cờ kết thúc → không phân biệt "từ" với "tiền tố".
- Tạo nút mới **đè** nút con đã có (`children[ch] = Node()` vô điều kiện) → mất nhánh cũ.
- Cờ kết thúc dùng khóa trùng với một ký tự hợp lệ.

## 2.2 Áp dụng vào bài này

Bài yêu cầu cài đặt chính cấu trúc **trie** ở trên: `insert` đi/tạo đường theo từng ký tự rồi đặt cờ kết thúc; `search` cần đường đi tồn tại **và** cờ kết thúc; `startsWith` chỉ cần đường đi tồn tại. Mọi thao tác `O(độ dài chuỗi)`.

# 3. Nền tảng: trie trông như thế nào?

Thêm các từ `app`, `apple`, `ape`, `bat`:

```text
            (gốc)
           /     \
          a       b
          |       |
          p       a
         / \      |
        p*  e*    t*
        |
        l
        |
        e*                 * = nút kết thúc một từ (is_end = True)
```

- Mỗi **nút** ứng với một **tiền tố** (đường đi từ gốc tới nó): nút `a → p → p` là tiền tố `"app"`.
- Các từ có chung tiền tố **dùng chung** đường đi → tiết kiệm và tra tiền tố rất nhanh.
- `is_end` đánh dấu "tiền tố này cũng là một từ đã thêm".

In [4]:
from typing import Dict, List, Optional
import random, time


class TrieNode:
    def __init__(self):
        self.children: Dict[str, "TrieNode"] = {}
        self.is_end = False


def show(node, prefix="", depth=0):
    """In cây theo dạng thụt lề."""
    for ch in sorted(node.children):
        child = node.children[ch]
        print("    " * depth + ch + ("*" if child.is_end else "") + f"   ({prefix + ch})")
        show(child, prefix + ch, depth + 1)

# 4. Baseline: lưu các từ trong `set`

In [5]:
class TrieSet:
    def __init__(self):
        self.words = set()
    def insert(self, word):
        self.words.add(word)
    def search(self, word):
        return word in self.words                                  # O(L) trung bình
    def startsWith(self, prefix):
        return any(w.startswith(prefix) for w in self.words)       # O(N · L) — duyệt mọi từ!

t = TrieSet()
for w in ["apple", "app", "bat"]:
    t.insert(w)
print(t.search("app"), t.startsWith("ba"), t.startsWith("c"))

True True False


`search` nhanh, nhưng `startsWith` phải **duyệt mọi từ** — với `3·10^4` từ và `3·10^4` truy vấn là quá chậm. Trie trả lời tiền tố trong `O(độ dài tiền tố)`.

# 5. Ý tưởng chính

```text
insert(word):  node = root; với mỗi ký tự: nếu chưa có con thì tạo; đi xuống. Cuối cùng node.is_end = True
_walk(s):      node = root; với mỗi ký tự: nếu không có con → None; đi xuống. Trả về node
search(word):  node = _walk(word); return node is not None and node.is_end
startsWith(p): return _walk(p) is not None
```

In [6]:
class TrieDict:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word: str) -> None:
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.is_end = True

    def _walk(self, s: str) -> Optional[TrieNode]:
        node = self.root
        for ch in s:
            node = node.children.get(ch)
            if node is None:
                return None
        return node

    def search(self, word: str) -> bool:
        node = self._walk(word)
        return node is not None and node.is_end

    def startsWith(self, prefix: str) -> bool:
        return self._walk(prefix) is not None


tr = TrieDict()
for w in ["app", "apple", "ape", "bat"]:
    tr.insert(w)
show(tr.root)
print("search('app') =", tr.search("app"), "| search('appl') =", tr.search("appl"), "| startsWith('appl') =", tr.startsWith("appl"))

a   (a)
    p   (ap)
        e*   (ape)
        p*   (app)
            l   (appl)
                e*   (apple)
b   (b)
    a   (ba)
        t*   (bat)
search('app') = True | search('appl') = False | startsWith('appl') = True


# 6. Mô phỏng

In [7]:
def trace_walk(trie, s):
    node = trie.root
    path = ""
    for ch in s:
        nxt = node.children.get(ch)
        if nxt is None:
            print(f"   '{path}' không có con '{ch}' → dừng: tiền tố '{s}' KHÔNG tồn tại")
            return
        path += ch
        node = nxt
        print(f"   đi '{ch}' → nút '{path}' (is_end = {node.is_end}, con: {sorted(node.children)})")
    print(f"   → startsWith = True, search = {node.is_end}")

for q in ["app", "ap", "apx"]:
    print(f"truy vấn '{q}':")
    trace_walk(tr, q)

truy vấn 'app':
   đi 'a' → nút 'a' (is_end = False, con: ['p'])
   đi 'p' → nút 'ap' (is_end = False, con: ['e', 'p'])
   đi 'p' → nút 'app' (is_end = True, con: ['l'])
   → startsWith = True, search = True
truy vấn 'ap':
   đi 'a' → nút 'a' (is_end = False, con: ['p'])
   đi 'p' → nút 'ap' (is_end = False, con: ['e', 'p'])
   → startsWith = True, search = False
truy vấn 'apx':
   đi 'a' → nút 'a' (is_end = False, con: ['p'])
   đi 'p' → nút 'ap' (is_end = False, con: ['e', 'p'])
   'ap' không có con 'x' → dừng: tiền tố 'apx' KHÔNG tồn tại


# 7. Cách lưu con bằng mảng 26 phần tử

Với bảng chữ cái cố định (26 chữ thường), mỗi nút có thể giữ `children = [None] * 26`, chỉ số `ord(ch) − ord('a')`. Truy cập con nhanh và đều, nhưng mỗi nút tốn 26 ô dù ít con. `dict` tiết kiệm bộ nhớ hơn khi cây thưa.

In [8]:
class TrieArray:
    __slots__ = ("root",)

    class Node:
        __slots__ = ("children", "is_end")
        def __init__(self):
            self.children = [None] * 26
            self.is_end = False

    def __init__(self):
        self.root = TrieArray.Node()

    def insert(self, word):
        node = self.root
        for ch in word:
            i = ord(ch) - 97
            if node.children[i] is None:
                node.children[i] = TrieArray.Node()
            node = node.children[i]
        node.is_end = True

    def _walk(self, s):
        node = self.root
        for ch in s:
            node = node.children[ord(ch) - 97]
            if node is None:
                return None
        return node

    def search(self, word):
        node = self._walk(word)
        return node is not None and node.is_end

    def startsWith(self, prefix):
        return self._walk(prefix) is not None

ta = TrieArray()
ta.insert("apple")
print(ta.search("apple"), ta.search("app"), ta.startsWith("app"))

True False True


# 8. Phiên bản đúng format LeetCode

In [9]:
class Trie:
    def __init__(self):
        self.root = {}                         # mỗi nút là một dict: ký tự → dict con; khóa "$" đánh dấu kết thúc

    def insert(self, word: str) -> None:
        node = self.root
        for ch in word:
            node = node.setdefault(ch, {})
        node["$"] = True

    def _walk(self, s: str):
        node = self.root
        for ch in s:
            if ch not in node:
                return None
            node = node[ch]
        return node

    def search(self, word: str) -> bool:
        node = self._walk(word)
        return node is not None and "$" in node

    def startsWith(self, prefix: str) -> bool:
        return self._walk(prefix) is not None

Bản này dùng **dict lồng nhau** thay cho lớp `TrieNode` — gọn hơn; ký tự `"$"` không phải chữ thường nên không trùng với cạnh nào.

In [10]:
def run_ops(cls, ops, args):
    obj, out = None, []
    for op, a in zip(ops, args):
        if op == "Trie":
            obj = cls(); out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out


def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


OPS = ["Trie", "insert", "search", "search", "startsWith", "insert", "search"]
ARGS = [[], ["apple"], ["apple"], ["app"], ["app"], ["app"], ["app"]]
EXP = [None, None, True, False, True, None, True]
check("Ví dụ (dict lồng)", run_ops(Trie, OPS, ARGS), EXP)
check("Ví dụ (TrieNode)", run_ops(TrieDict, OPS, ARGS), EXP)
check("Ví dụ (mảng 26)", run_ops(TrieArray, OPS, ARGS), EXP)

random.seed(208)
for _ in range(300):
    ops, args = ["Trie"], [[]]
    for _ in range(random.randint(1, 30)):
        op = random.choice(["insert", "search", "startsWith"])
        w = "".join(random.choice("ab") for _ in range(random.randint(1, 4)))
        ops.append(op); args.append([w])
    exp = run_ops(TrieSet, ops, args)
    assert run_ops(Trie, ops, args) == exp == run_ops(TrieDict, ops, args) == run_ops(TrieArray, ops, args)
print("300 chuỗi thao tác ngẫu nhiên: ba cài đặt khớp bản set")

words = ["".join(random.choice("abcdefghij") for _ in range(random.randint(5, 12))) for _ in range(15000)]
prefixes = [w[:random.randint(1, 4)] for w in random.sample(words, 2000)]
for name, cls in [("Trie", Trie), ("TrieSet", TrieSet)]:
    obj = cls()
    for w in words:
        obj.insert(w)
    t0 = time.time()
    for p in prefixes:
        obj.startsWith(p)
    t1 = time.time()
    print(f"{name:<8}: 2000 truy vấn startsWith trên 15000 từ mất {t1 - t0:.3f}s")

Ví dụ (dict lồng) PASS  [None, None, True, False, True, None, True]
Ví dụ (TrieNode) PASS  [None, None, True, False, True, None, True]
Ví dụ (mảng 26)  PASS  [None, None, True, False, True, None, True]
300 chuỗi thao tác ngẫu nhiên: ba cài đặt khớp bản set


Trie    : 2000 truy vấn startsWith trên 15000 từ mất 0.001s

TrieSet : 2000 truy vấn startsWith trên 15000 từ mất 0.346s


# 9. Vì sao đúng? Độ phức tạp?

**Bất biến:** với mọi chuỗi `s`, `_walk(s)` trả về một nút ⟺ `s` là tiền tố của **ít nhất một** từ đã thêm (mỗi lần `insert` tạo đủ đường đi cho mọi tiền tố của từ); và nút đó có cờ kết thúc ⟺ `s` chính là một từ đã thêm.

Với `L` là độ dài từ/tiền tố, `S` là tổng độ dài các từ đã thêm:

```text
insert:     O(L)
search:     O(L)
startsWith: O(L)          (không phụ thuộc số từ!)
Bộ nhớ:     O(S) nút (dict) — mảng 26: O(26 · S) ô
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Không có cờ kết thúc — `search` thành `startsWith`

In [11]:
class WrongNoEnd(TrieDict):
    def search(self, word):
        return self._walk(word) is not None              # SAI: tiền tố cũng được coi là từ

print("Sai :", run_ops(WrongNoEnd, OPS, ARGS))
print("Đúng:", EXP)

Sai : [None, None, True, True, True, None, True]
Đúng: [None, None, True, False, True, None, True]


## Lỗi 2: Đặt `is_end` cho **mọi** nút trên đường đi

In [12]:
class WrongEndEverywhere(TrieDict):
    def insert(self, word):
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
            node.is_end = True                            # SAI

print("Sai :", run_ops(WrongEndEverywhere, OPS, ARGS))

Sai : [None, None, True, True, True, None, True]


## Lỗi 3: Tạo nút mới **đè** lên nút con đã có

In [13]:
class WrongOverwrite(TrieDict):
    def insert(self, word):
        node = self.root
        for ch in word:
            node.children[ch] = TrieNode()               # SAI: xóa mất cả nhánh cũ
            node = node.children[ch]
        node.is_end = True

w3 = WrongOverwrite()
w3.insert("apple"); w3.insert("app")
print("Sau khi thêm 'apple' rồi 'app': search('apple') =", w3.search("apple"), "(đúng phải là True)")

Sau khi thêm 'apple' rồi 'app': search('apple') = False (đúng phải là True)


## Lỗi 4: `startsWith` trả lời bằng cờ kết thúc

`startsWith("app")` sau khi chỉ thêm `"apple"` phải là `True` dù nút `"app"` không có cờ. `startsWith` chỉ cần **đường đi tồn tại**.

# 11. Mẹo nhận diện pattern

- **Nhiều truy vấn tiền tố trên một tập từ** → trie.
- **Tìm kiếm có ký tự đại diện / tìm nhiều từ trong lưới** → trie + DFS (#211, #212).
- **Đếm theo tiền tố, gợi ý tự động** → lưu thêm thông tin ở mỗi nút (số từ đi qua, danh sách gợi ý).
- **Trie trên bit** → bài XOR lớn nhất (#421).

Bài liên quan: **#211 Design Add and Search Words Data Structure**, **#212 Word Search II** (Hard), **#648 Replace Words**, **#677 Map Sum Pairs**, **#1268 Search Suggestions System**, **#421 Maximum XOR of Two Numbers in an Array**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Vẽ trie sau khi thêm `"to"`, `"tea"`, `"ted"`, `"ten"`, `"i"`, `"in"`, `"inn"`. Đánh dấu các nút kết thúc.

## Bài 2 — Điền chỗ trống

In [14]:
class Exercise2Trie:
    def __init__(self):
        self.root = {}

    def insert(self, word: str) -> None:
        node = self.root
        for ch in word:
            node = node.setdefault(ch, {})
        node["$"] = True

    def search(self, word: str) -> bool:
        # TODO
        pass

    def startsWith(self, prefix: str) -> bool:
        # TODO
        pass


# assert run_ops(Exercise2Trie, ["Trie", "insert", "search", "search", "startsWith", "insert", "search"], [[], ["apple"], ["apple"], ["app"], ["app"], ["app"], ["app"]]) == [None, None, True, False, True, None, True]

## Bài 3 — Biến thể: đếm số từ có tiền tố cho trước

Thêm `count_prefix(prefix)`: số từ đã thêm (tính cả lặp lại) bắt đầu bằng `prefix`. Gợi ý: mỗi nút giữ `pass_count` = số từ đi qua nút.

In [15]:
class Exercise3CountTrie:
    def __init__(self):
        # TODO
        pass

    def insert(self, word: str) -> None:
        pass

    def count_prefix(self, prefix: str) -> int:
        pass


# c3 = Exercise3CountTrie(); [c3.insert(w) for w in ["apple", "app", "apt", "bat", "app"]]; assert c3.count_prefix("ap") == 4 and c3.count_prefix("app") == 3 and c3.count_prefix("b") == 1 and c3.count_prefix("c") == 0

## Bài 4 — Biến thể: Replace Words (LeetCode #648)

Cho danh sách **gốc từ** và một câu; thay mỗi từ trong câu bằng gốc **ngắn nhất** là tiền tố của nó (nếu có).

In [16]:
def exercise_4_replace_words(dictionary: List[str], sentence: str) -> str:
    # TODO
    pass


# assert exercise_4_replace_words(["cat", "bat", "rat"], "the cattle was rattled by the battery") == "the cat was rat by the bat"
# assert exercise_4_replace_words(["a", "b", "c"], "aadsfasf absbs bbab cadsfafs") == "a a b c"

## Bài 5 — Mini challenge: Search Suggestions System (LeetCode #1268)

Với mỗi tiền tố `searchWord[:i]` (`i = 1..len`), trả về tối đa **3** từ trong `products` có tiền tố đó, theo thứ tự từ điển. Gợi ý: sắp `products` trước, mỗi nút trie lưu tối đa 3 từ đầu tiên đi qua nó.

In [17]:
def exercise_5_suggested_products(products: List[str], searchWord: str) -> List[List[str]]:
    # TODO
    pass


# assert exercise_5_suggested_products(["mobile", "mouse", "moneypot", "monitor", "mousepad"], "mouse") == [["mobile", "moneypot", "monitor"], ["mobile", "moneypot", "monitor"], ["mouse", "mousepad"], ["mouse", "mousepad"], ["mouse", "mousepad"]]
# assert exercise_5_suggested_products(["havana"], "tatiana") == [[], [], [], [], [], [], []]

# 13. Gợi ý

**Bài 1**
- Mức 1: thêm các từ vào `TrieDict` rồi `show(...)` sau khi tự vẽ.

**Bài 3**
- Mức 1: nút là dict có thêm khóa `"#"` lưu `pass_count`; `insert` tăng `"#"` ở mọi nút đi qua.

**Bài 4**
- Mức 1: với mỗi từ, đi trong trie từng ký tự; gặp nút kết thúc đầu tiên → trả tiền tố đó; rơi khỏi trie → giữ nguyên từ.

**Bài 5**
- Mức 1: `products.sort()`; khi `insert`, ở mỗi nút đi qua, nếu danh sách gợi ý < 3 thì thêm từ.
- Mức 2: khi tra, nếu rơi khỏi trie thì các tiền tố dài hơn đều có `[]`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [18]:
t1 = TrieDict()
for w in ["to", "tea", "ted", "ten", "i", "in", "inn"]:
    t1.insert(w)
show(t1.root)

i*   (i)
    n*   (in)
        n*   (inn)
t   (t)
    e   (te)
        a*   (tea)
        d*   (ted)
        n*   (ten)
    o*   (to)


## Đáp án Bài 2

In [19]:
class Answer2Trie(Trie):
    pass

assert run_ops(Answer2Trie, ["Trie", "insert", "search", "search", "startsWith", "insert", "search"], [[], ["apple"], ["apple"], ["app"], ["app"], ["app"], ["app"]]) == [None, None, True, False, True, None, True]
print("Bài 2 PASS (giống Trie ở mục 8)")

Bài 2 PASS (giống Trie ở mục 8)


## Đáp án Bài 3

In [20]:
class Answer3CountTrie:
    def __init__(self):
        self.root = {"#": 0}

    def insert(self, word: str) -> None:
        node = self.root
        node["#"] += 1
        for ch in word:
            node = node.setdefault(ch, {"#": 0})
            node["#"] += 1

    def count_prefix(self, prefix: str) -> int:
        node = self.root
        for ch in prefix:
            if ch not in node:
                return 0
            node = node[ch]
        return node["#"]

c3 = Answer3CountTrie(); [c3.insert(w) for w in ["apple", "app", "apt", "bat", "app"]]; assert c3.count_prefix("ap") == 4 and c3.count_prefix("app") == 3 and c3.count_prefix("b") == 1 and c3.count_prefix("c") == 0
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [21]:
def answer_4_replace_words(dictionary: List[str], sentence: str) -> str:
    trie = Trie()
    for r in dictionary:
        trie.insert(r)
    out = []
    for w in sentence.split():
        node, rep = trie.root, w
        for i, ch in enumerate(w):
            if ch not in node:
                break
            node = node[ch]
            if "$" in node:
                rep = w[:i + 1]
                break
        out.append(rep)
    return " ".join(out)

assert answer_4_replace_words(["cat", "bat", "rat"], "the cattle was rattled by the battery") == "the cat was rat by the bat"
assert answer_4_replace_words(["a", "b", "c"], "aadsfasf absbs bbab cadsfafs") == "a a b c"
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [22]:
def answer_5_suggested_products(products: List[str], searchWord: str) -> List[List[str]]:
    root = {}
    for p in sorted(products):
        node = root
        for ch in p:
            node = node.setdefault(ch, {})
            sug = node.setdefault("$s", [])
            if len(sug) < 3:
                sug.append(p)
    out, node = [], root
    for ch in searchWord:
        node = node.get(ch) if node is not None else None
        out.append(list(node["$s"]) if node is not None else [])
    return out

assert answer_5_suggested_products(["mobile", "mouse", "moneypot", "monitor", "mousepad"], "mouse") == [["mobile", "moneypot", "monitor"], ["mobile", "moneypot", "monitor"], ["mouse", "mousepad"], ["mouse", "mousepad"], ["mouse", "mousepad"]]
assert answer_5_suggested_products(["havana"], "tatiana") == [[], [], [], [], [], [], []]
for _ in range(500):
    prods = ["".join(random.choice("ab") for _ in range(random.randint(1, 4))) for _ in range(random.randint(1, 8))]
    sw = "".join(random.choice("ab") for _ in range(random.randint(1, 4)))
    exp = [sorted(p for p in prods if p.startswith(sw[:i]))[:3] for i in range(1, len(sw) + 1)]
    assert answer_5_suggested_products(prods, sw) == exp
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> Trie: mỗi nút là một tiền tố, con theo từng ký tự (dict hoặc mảng 26), cờ kết thúc đánh dấu từ hoàn chỉnh. `insert` tạo đường đi (không đè nút cũ), `search` cần đường đi **và** cờ, `startsWith` chỉ cần đường đi. Mọi thao tác `O(độ dài)`.

---

## Checklist trước khi nộp

- [ ] Có cờ kết thúc, chỉ đặt ở nút cuối của từ?
- [ ] `insert` dùng `setdefault`/kiểm tra trước khi tạo nút?
- [ ] `search` kiểm tra cờ; `startsWith` thì không?
- [ ] Ký hiệu cờ (`"$"`) không trùng ký tự hợp lệ?